In [60]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
import optuna
from torch.optim import SGD,Adam,RMSprop
torch.cuda.is_available()
torch.cuda.get_device_name()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [20]:
train_df=pd.read_csv("/kaggle/input/datasets/xiaoqingjiang/fashionmnist/fashion-mnist_train.csv")
test_df=pd.read_csv("/kaggle/input/datasets/xiaoqingjiang/fashionmnist/fashion-mnist_test.csv")

In [21]:
train_df.shape

(60000, 785)

In [22]:
train_df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
y_train=train_df.label
X_train=train_df.iloc[:,1:]
y_test=test_df.label
X_test=test_df.iloc[:,1:]

In [24]:
X_train=X_train/225.0
X_test=X_test/225.0
X_train.head()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.022222,0.0,0.0,...,0.000000,0.0,0.0,0.133333,0.191111,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.004444,0.008889,0.0,0.0,0.000000,0.0,0.0,...,0.013333,0.0,0.0,0.000000,0.000000,0.004444,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0


In [25]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features=torch.tensor(features.to_numpy(),dtype=torch.float32)
        self.labels=torch.tensor(labels,dtype=torch.float32)
    def __len__(self):
        return self.features.shape[0]
    def __getitem__(self,item):
        return self.features[item],self.labels[item]

In [28]:
train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)

In [29]:
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [41]:
features, labels = next(iter(train_dataloader))
print(features.shape)

torch.Size([32, 784])


In [61]:
class MyModel(nn.Module):
    def __init__(self,num_features,num_layers,num_nodes_per_layer,num_out,dropout_nodes):
        super().__init__()
        self.Layers=[]
        for _ in range(num_layers):
            self.Layers.append(nn.Linear(num_features,num_nodes_per_layer))
            self.Layers.append(nn.BatchNorm1d(num_nodes_per_layer))
            self.Layers.append(nn.ReLU())
            self.Layers.append(nn.Dropout(dropout_nodes))
            num_features=num_nodes_per_layer
        self.Layers.append(nn.Linear(num_nodes_per_layer,num_out))
        self.model=nn.Sequential(*self.Layers)

    def forward(self,features):
        return self.model(features)
        

In [66]:

def objective(trial):
    num_layers = trial.suggest_int("num_layers", 1, 6)
    num_nodes = trial.suggest_int(
        "num_nodes_per_layer", 8, 128, step=8
    )
    epochs = trial.suggest_categorical(
        "epochs",[10,20,30,40,50]
    )
    
    optimizer_name = trial.suggest_categorical(
        "optimizer", ["SGD", "Adam","Rmsprop"]
    )
    batch_size = trial.suggest_int(
        "batch_size", 32, 128, step=32
    )
    learning_rate = trial.suggest_float(
        "learning_rate", 1e-4, 1e-1, log=True
    )
    weight_decay= trial.suggest_float(
        "weight_decay", 1e-5, 1e-3, log=True
    )
    dropout_nodes=trial.suggest_float(
        "dropout_nodes",0.1,0.5,step=0.1
    )
    model = MyModel(
        num_features=784, 
        num_layers=num_layers,
        num_nodes_per_layer=num_nodes,
        num_out=10,
        dropout_nodes=dropout_nodes
    )
    model.to(device)
    if optimizer_name == "SGD":
        optimizer = SGD(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
    elif optimizer_name=="Adam":
        optimizer = Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
    else:
        optimizer = RMSprop(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):

        for features, labels in train_loader:
            features = features.view(features.shape[0],-1).to(device, dtype=torch.float32)
            labels = labels.to(device, dtype=torch.long)


            optimizer.zero_grad()

            logits = model(features)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

    
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in val_loader:
            features = features.view(features.shape[0],-1).to(device, dtype=torch.float32)
            labels = labels.to(device, dtype=torch.long)

            logits = model(features)
            predictions = logits.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()
            total += labels.size(0)

        validation_accuracy = correct / total

    return validation_accuracy

In [68]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

[I 2026-11-06 20:19:22,323] A new study created in memory with name: no-name-a69459d6-fd22-4bcd-9050-ca1703c70ad4
[I 2026-11-06 20:19:43,565] Trial 0 finished with value: 0.8441 and parameters: {'num_layers': 2, 'num_nodes_per_layer': 16, 'epochs': 10, 'optimizer': 'SGD', 'batch_size': 128, 'learning_rate': 0.02129855599706798, 'weight_decay': 0.00011793281141382365, 'dropout_nodes': 0.2}. Best is trial 0 with value: 0.8441.
[I 2026-11-06 20:21:27,198] Trial 1 finished with value: 0.8344 and parameters: {'num_layers': 4, 'num_nodes_per_layer': 16, 'epochs': 40, 'optimizer': 'Rmsprop', 'batch_size': 128, 'learning_rate': 0.00014592147674934514, 'weight_decay': 0.0001417951979979293, 'dropout_nodes': 0.2}. Best is trial 0 with value: 0.8441.
[I 2026-11-06 20:22:15,120] Trial 2 finished with value: 0.8628 and parameters: {'num_layers': 2, 'num_nodes_per_layer': 80, 'epochs': 10, 'optimizer': 'SGD', 'batch_size': 32, 'learning_rate': 0.06813808337432609, 'weight_decay': 0.00093719427532688

In [70]:
study.best_params

{'num_layers': 2,
 'num_nodes_per_layer': 80,
 'epochs': 30,
 'optimizer': 'Rmsprop',
 'batch_size': 96,
 'learning_rate': 0.000497080249899111,
 'weight_decay': 0.00018605034308466248,
 'dropout_nodes': 0.2}

In [79]:
num_layers = 2 
num_nodes = 80 
epochs= 30
optimizer= 'Rmsprop'
batch_size= 96
learning_rate=0.000497080249899111
weight_decay= 0.00018605034308466248
dropout_nodes= 0.2

In [80]:
num_layers

2

In [83]:
model = MyModel(
        num_features=784, 
        num_layers=num_layers,
        num_nodes_per_layer=num_nodes,
        num_out=10,
        dropout_nodes=dropout_nodes
    )
model.to(device)
optimizer = RMSprop(model.parameters(), lr=learning_rate,weight_decay=weight_decay)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)
criterion = nn.CrossEntropyLoss()
for epoch in range(epochs):
    for features, labels in train_loader:
        features = features.view(features.shape[0],-1).to(device, dtype=torch.float32)
        labels = labels.to(device, dtype=torch.long)

        optimizer.zero_grad()

        logits = model(features)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

    
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for features, labels in val_loader:
        features = features.view(features.shape[0],-1).to(device, dtype=torch.float32)
        labels = labels.to(device, dtype=torch.long)

        logits = model(features)
        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)

    validation_accuracy = correct / total

print(validation_accuracy)

0.8925


In [84]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for features, labels in train_loader:
        features = features.view(features.shape[0],-1).to(device, dtype=torch.float32)
        labels = labels.to(device, dtype=torch.long)

        logits = model(features)
        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)

    train_accuracy = correct / total

print(train_accuracy)

0.9267333333333333
